In [1]:
import os
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 2 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
    print(f"Using GPU {gpu_num}")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)
# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

import pandas as pd
from data_preprocessing import preprocess_data_channel
from mlp.mlp_model import train_mlp_channel
from evaluation import print_metrics_multiclass, get_metrics_multiclass
from visualization import plot_confusion_matrix_multiclass, plot_roc_curve_multiclass
from ml_algorithms import train_knn_channel, train_svm_channel, train_random_forest_channel, train_xgboost



Using GPU 2


E0000 00:00:1747145822.529377  366719 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747145822.535172  366719 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Using GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


Full Dataset NLOS/LOS Classification
====================================

In [2]:
# Load the datasets
# df_tdl_cdl = pd.read_csv("dataset_tdl_cdl_training_siso.csv")
# df_tdl_cdl_test = pd.read_csv("dataset_tdl_cdl_testing.csv")

df_tdl_cdl_siso = pd.read_csv("dataset_tdl_cdl_training_siso.csv")
df_tdl_cdl_siso_test = pd.read_csv("dataset_tdl_cdl_testing.csv")
df_tdl_cdl_mimo = pd.read_csv("dataset_tdl_cdl_training_mimo.csv")
df_tdl_cdl_mimo_test = pd.read_csv("dataset_tdl_cdl_testing_mimo.csv")
df_tdl_cdl_0db_siso_test = pd.read_csv("dataset_tdl_cdl_0db_siso.csv")
df_tdl_cdl_0db_mimo_test = pd.read_csv("dataset_tdl_cdl_0db_mimo.csv")
# 5db
df_tdl_cdl_5db_siso_test = pd.read_csv("dataset_tdl_cdl_5db_siso.csv")
df_tdl_cdl_5db_mimo_test = pd.read_csv("dataset_tdl_cdl_5db_mimo.csv")
# 10db
df_tdl_cdl_10db_siso_test = pd.read_csv("dataset_tdl_cdl_10db_siso.csv")
df_tdl_cdl_10db_mimo_test = pd.read_csv("dataset_tdl_cdl_10db_mimo.csv")
# 15db
df_tdl_cdl_15db_siso_test = pd.read_csv("dataset_tdl_cdl_15db_siso.csv")
df_tdl_cdl_15db_mimo_test = pd.read_csv("dataset_tdl_cdl_15db_mimo.csv")

# df_sionna_mimo_los = pd.read_csv("dataset/dataset_sionna_mimo_los.csv")
# df_sionna_mimo_los_test = pd.read_csv("dataset/dataset_sionna_mimo_los_test.csv")
# df_sionna_mimo_nlos = pd.read_csv("dataset_sionna_mimo_nlos.csv")
# df_sionna_mimo_nlos_test = pd.read_csv("dataset_sionna_mimo_nlos_test.csv")
# df_sionna_siso_los = pd.read_csv("dataset_sionna_siso_los.csv")
# df_sionna_siso_los_test = pd.read_csv("dataset_sionna_siso_los_test.csv")
# df_sionna_siso_nlos = pd.read_csv("temporary_data/dataset_sionna_siso_nlos.csv")
# df_sionna_siso_nlos_test = pd.read_csv("temporary_data/dataset_sionna_siso_nlos_test.csv")


# Make the dataset for tdl only and cdl only (based on the column name model_type)
# df_tdl = df_tdl_cdl[df_tdl_cdl['model_type'] == 'tdl']
# df_cdl = df_tdl_cdl[df_tdl_cdl['model_type'] == 'cdl']
# df_tdl_test = df_tdl_cdl_test[df_tdl_cdl_test['model_type'] == 'tdl']
# df_cdl_test = df_tdl_cdl_test[df_tdl_cdl_test['model_type'] == 'cdl']
# nlos_test.csv")

# # concatenate the datasets
# df_sionna_mimo = pd.concat([df_sionna_mimo_los, df_sionna_mimo_nlos])
# df_sionna_mimo_test = pd.concat([df_sionna_mimo_los_test, df_sionna_mimo_nlos_test])



# # concatenate the datasets
# df_sionna_siso = pd.concat([df_sionna_siso_los, df_sionna_siso_nlos])
# df_sionna_siso_test = pd.concat([df_sionna_siso_los_test, df_sionna_siso_nlos_test])



df_tdl_siso = df_tdl_cdl_siso[df_tdl_cdl_siso['model_type'] == 'tdl']
df_cdl_siso = df_tdl_cdl_siso[df_tdl_cdl_siso['model_type'] == 'cdl']
df_tdl_siso_test = df_tdl_cdl_siso_test[df_tdl_cdl_siso_test['model_type'] == 'tdl']
df_cdl_siso_test = df_tdl_cdl_siso_test[df_tdl_cdl_siso_test['model_type'] == 'cdl']
df_tdl_mimo = df_tdl_cdl_mimo[df_tdl_cdl_mimo['model_type'] == 'tdl']
df_cdl_mimo = df_tdl_cdl_mimo[df_tdl_cdl_mimo['model_type'] == 'cdl']
df_tdl_mimo_test = df_tdl_cdl_mimo_test[df_tdl_cdl_mimo_test['model_type'] == 'tdl']
df_cdl_mimo_test = df_tdl_cdl_mimo_test[df_tdl_cdl_mimo_test['model_type'] == 'cdl']
df_tdl_0db_siso_test = df_tdl_cdl_0db_siso_test[df_tdl_cdl_0db_siso_test['model_type'] == 'tdl']
df_cdl_0db_siso_test = df_tdl_cdl_0db_siso_test[df_tdl_cdl_0db_siso_test['model_type'] == 'cdl']
df_tdl_0db_mimo_test = df_tdl_cdl_0db_mimo_test[df_tdl_cdl_0db_mimo_test['model_type'] == 'tdl']
df_cdl_0db_mimo_test = df_tdl_cdl_0db_mimo_test[df_tdl_cdl_0db_mimo_test['model_type'] == 'cdl']
df_tdl_5db_siso_test = df_tdl_cdl_5db_siso_test[df_tdl_cdl_5db_siso_test['model_type'] == 'tdl']
df_cdl_5db_siso_test = df_tdl_cdl_5db_siso_test[df_tdl_cdl_5db_siso_test['model_type'] == 'cdl']
df_tdl_5db_mimo_test = df_tdl_cdl_5db_mimo_test[df_tdl_cdl_5db_mimo_test['model_type'] == 'tdl']
df_cdl_5db_mimo_test = df_tdl_cdl_5db_mimo_test[df_tdl_cdl_5db_mimo_test['model_type'] == 'cdl']
df_tdl_10db_siso_test = df_tdl_cdl_10db_siso_test[df_tdl_cdl_10db_siso_test['model_type'] == 'tdl']
df_cdl_10db_siso_test = df_tdl_cdl_10db_siso_test[df_tdl_cdl_10db_siso_test['model_type'] == 'cdl']
df_tdl_10db_mimo_test = df_tdl_cdl_10db_mimo_test[df_tdl_cdl_10db_mimo_test['model_type'] == 'tdl']
df_cdl_10db_mimo_test = df_tdl_cdl_10db_mimo_test[df_tdl_cdl_10db_mimo_test['model_type'] == 'cdl']
df_tdl_15db_siso_test = df_tdl_cdl_15db_siso_test[df_tdl_cdl_15db_siso_test['model_type'] == 'tdl']
df_cdl_15db_siso_test = df_tdl_cdl_15db_siso_test[df_tdl_cdl_15db_siso_test['model_type'] == 'cdl']
df_tdl_15db_mimo_test = df_tdl_cdl_15db_mimo_test[df_tdl_cdl_15db_mimo_test['model_type'] == 'tdl']
df_cdl_15db_mimo_test = df_tdl_cdl_15db_mimo_test[df_tdl_cdl_15db_mimo_test['model_type'] == 'cdl']

In [3]:
import pandas as pd
from evaluation import get_metrics_multiclass
df_full = df_tdl_cdl_siso
df_full_test_list = [
    ("-3-20db", df_tdl_cdl_siso_test),
    ("0db", df_tdl_cdl_0db_siso_test),
    ("5db", df_tdl_cdl_5db_siso_test),
    ("10db", df_tdl_cdl_10db_siso_test),
    ("15db", df_tdl_cdl_15db_siso_test),
]
cases = [
    {
        "name": "TDL SISO",
        "train": df_tdl_siso,
        "test": df_tdl_siso_test,
        "snr_tests": [
            ("0db", df_tdl_0db_siso_test),
            ("5db", df_tdl_5db_siso_test),
            ("10db", df_tdl_10db_siso_test),
            ("15db", df_tdl_15db_siso_test),
        ],
    },
    {
        "name": "CDL SISO",
        "train": df_cdl_siso,
        "test": df_cdl_siso_test,
        "snr_tests": [
            ("0db", df_cdl_0db_siso_test),
            ("5db", df_cdl_5db_siso_test),
            ("10db", df_cdl_10db_siso_test),
            ("15db", df_cdl_15db_siso_test),
        ],
    },
    {
        "name": "TDL MIMO",
        "train": df_tdl_mimo,
        "test": df_tdl_mimo_test,
        "snr_tests": [
            ("0db", df_tdl_0db_mimo_test),
            ("5db", df_tdl_5db_mimo_test),
            ("10db", df_tdl_10db_mimo_test),
            ("15db", df_tdl_15db_mimo_test),
        ],
    },
    {
        "name": "CDL MIMO",
        "train": df_cdl_mimo,
        "test": df_cdl_mimo_test,
        "snr_tests": [
            ("0db", df_cdl_0db_mimo_test),
            ("5db", df_cdl_5db_mimo_test),
            ("10db", df_cdl_10db_mimo_test),
            ("15db", df_cdl_15db_mimo_test),
        ],
    },
]


results = []

for case in cases:
    features, labels = preprocess_data_channel(case["train"])
    features_test, labels_test = preprocess_data_channel(case["test"])
    y_train, y_train_pred, y_train_prob, y_test, y_test_pred, y_test_prob, mlp = train_mlp_channel(features, labels, features_test, labels_test)
    metrics = get_metrics_multiclass(y_test, y_test_pred, y_test_prob)
    metrics['Case'] = case['name']
    metrics['SNR'] = '-3-20db'
    results.append(metrics)
    for snr_name, df_snr in case["snr_tests"]:
        features_snr, labels_snr = preprocess_data_channel(df_snr)
        # y_test_pred = mlp.predict(features_snr)
        y_test_pred = mlp.predict(features_snr).argmax(axis=1)
        # y_test_prob = mlp.predict_proba(features_snr)[:, 1]
        y_test_prob = mlp.predict(features_snr)
        # metrics = get_metrics_multiclass(labels_snr, y_test_pred, y_test_prob)
        metrics = get_metrics_multiclass(labels_snr, y_test_pred, y_test_prob)
        metrics['Case'] = case['name']
        metrics['SNR'] = snr_name
        results.append(metrics)

df_results = pd.DataFrame(results)
cols = ['Case', 'SNR'] + [col for col in df_results.columns if col not in ['Case', 'SNR']]
df_results = df_results[cols]


/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_abs"] = features["mean"].apply(lambda x: abs(complex(x)))
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_phase"] = features["mean"].apply(lambda x: cmath.phase(complex(x)))    # remove the original mean column
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to b

Epoch 1/1000


I0000 00:00:1747145842.344194  367111 service.cc:148] XLA service 0x7fe1e0007e40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1747145842.344265  367111 service.cc:156]   StreamExecutor device (0): Tesla V100-PCIE-16GB, Compute Capability 7.0
I0000 00:00:1747145842.505067  367111 cuda_dnn.cc:529] Loaded cuDNN version 90300


 74/313 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.2350 - loss: 1.5783

I0000 00:00:1747145843.219618  367111 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


...

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 948us/step
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 968us/step
114/157 ━━━━━━━━━━━━━━━━━━━━ 0s 889us/step

/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_abs"] = features["mean"].apply(lambda x: abs(complex(x)))
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_phase"] = features["mean"].apply(lambda x: cmath.phase(complex(x)))    # remove the original mean column


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 982us/step
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 958us/step
117/157 ━━━━━━━━━━━━━━━━━━━━ 0s 871us/step

/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_abs"] = features["mean"].apply(lambda x: abs(complex(x)))
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_phase"] = features["mean"].apply(lambda x: cmath.phase(complex(x)))    # remove the original mean column


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 939us/step
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 925us/step


In [ ]:
print(df_results.to_string(index=False))
# save the results to a csv and xlsx file
df_results.to_csv("results.csv", index=False)
# df_results.to_excel("/results.xlsx", index=False)

    Case     SNR  Accuracy  Precision  Recall  F1 Score  ROC AUC  Log Loss
TDL SISO -3-20db    0.3410   0.338421  0.3410  0.338792 0.683709  1.710681
TDL SISO     0db    0.3196   0.324717  0.3196  0.319441 0.674793  1.676933
TDL SISO     5db    0.3224   0.320963  0.3224  0.318901 0.682631  1.634333
TDL SISO    10db    0.3356   0.333622  0.3356  0.333267 0.688069  1.665574
TDL SISO    15db    0.3232   0.318364  0.3232  0.319442 0.685589  1.733954
CDL SISO -3-20db    0.3242   0.332738  0.3242  0.325579 0.654058  1.666747
CDL SISO     0db    0.2982   0.294169  0.2982  0.293950 0.616167  1.718713
CDL SISO     5db    0.2916   0.292744  0.2916  0.288164 0.625091  1.643942
CDL SISO    10db    0.3344   0.342600  0.3344  0.331915 0.661078  1.640481
CDL SISO    15db    0.3782   0.388403  0.3782  0.379297 0.700543  1.587995
TDL MIMO -3-20db    0.3332   0.328131  0.3332  0.329790 0.715368  1.536785
TDL MIMO     0db    0.2782   0.276210  0.2782  0.274930 0.623664  1.623174
TDL MIMO     5db    0.349

ModuleNotFoundError: No module named 'openpyxl'

In [5]:
import pandas as pd
from evaluation import get_metrics_multiclass

classifiers = {
    "MLP": train_mlp_channel,
    "kNN": train_knn_channel,
    "SVM": train_svm_channel,
    "RF": train_random_forest_channel,
}

results = []

for case in cases:
    features, labels = preprocess_data_channel(case["train"])
    features_test, labels_test = preprocess_data_channel(case["test"])
    for clf_name, clf_func in classifiers.items():
        print(f"Training {clf_name} for {case['name']}")
        y_train, y_train_pred, y_train_prob, y_test, y_test_pred, y_test_prob, clf = clf_func(features, labels, features_test, labels_test)
        # metrics = get_metrics_multiclass(y_test, y_test_pred, y_test_prob)
        metrics = get_metrics_multiclass(y_test, y_test_pred, y_test_prob)
        metrics['Case'] = case['name']
        metrics['SNR'] = '-3-20db'
        metrics['Classifier'] = clf_name
        results.append(metrics)
        for snr_name, df_snr in case["snr_tests"]:
            features_snr, labels_snr = preprocess_data_channel(df_snr)
            if clf_name == "MLP":
                y_test_pred = clf.predict(features_snr).argmax(axis=1)
                y_test_prob = clf.predict(features_snr)
            else:
                y_test_pred = clf.predict(features_snr)
                y_test_prob = clf.predict_proba(features_snr) if hasattr(clf, "predict_proba") else None
            # metrics = get_metrics_multiclass(labels_snr, y_test_pred, y_test_prob)
            metrics = get_metrics_multiclass(labels_snr, y_test_pred, y_test_prob)
            metrics['Case'] = case['name']
            metrics['SNR'] = snr_name
            metrics['Classifier'] = clf_name
            results.append(metrics)

df_results = pd.DataFrame(results)
cols = ['Case', 'SNR', 'Classifier'] + [col for col in df_results.columns if col not in ['Case', 'SNR', 'Classifier']]
df_results = df_results[cols]


Training MLP for TDL SISO
Epoch 1/1000


/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_abs"] = features["mean"].apply(lambda x: abs(complex(x)))
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_phase"] = features["mean"].apply(lambda x: cmath.phase(complex(x)))    # remove the original mean column
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to b

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2815 - loss: 1.5318
Epoch 1: saving model to mlp_checkpoints/mlp_model_epoch_01.h5


313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.2815 - loss: 1.5317 - val_accuracy: 0.3160 - val_loss: 1.4373 - learning_rate: 0.0010
Epoch 2/1000
305/313 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3092 - loss: 1.4421
Epoch 2: saving model to mlp_checkpoints/mlp_model_epoch_02.h5


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3093 - loss: 1.4420 - val_accuracy: 0.3358 - val_loss: 1.4045 - learning_rate: 0.0010
Epoch 3/1000
291/313 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3149 - loss: 1.4231
Epoch 3: saving model to mlp_checkpoints/mlp_model_epoch_03.h5


...

/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_abs"] = features["mean"].apply(lambda x: abs(complex(x)))
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_phase"] = features["mean"].apply(lambda x: cmath.phase(complex(x)))    # remove the original mean column
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to b

Training SVM for CDL MIMO


/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_abs"] = features["mean"].apply(lambda x: abs(complex(x)))
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_phase"] = features["mean"].apply(lambda x: cmath.phase(complex(x)))    # remove the original mean column
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to b

Training RF for CDL MIMO
Performing cross-validation...
Cross-Validation Accuracy: 0.3959 ± 0.0140


/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_abs"] = features["mean"].apply(lambda x: abs(complex(x)))
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_phase"] = features["mean"].apply(lambda x: cmath.phase(complex(x)))    # remove the original mean column
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to b

In [6]:
print(df_results.to_string(index=False))
# save the results to a csv and xlsx file
df_results.to_csv("results1.csv", index=False)
# df_results.to_excel("results1.xlsx", index=False)

    Case     SNR Classifier  Accuracy  Precision  Recall  F1 Score  ROC AUC  Log Loss
TDL SISO -3-20db        MLP    0.3250   0.316098  0.3250  0.318498 0.689012  1.655967
TDL SISO     0db        MLP    0.3212   0.317482  0.3212  0.318443 0.669025  1.718747
TDL SISO     5db        MLP    0.3312   0.324676  0.3312  0.326122 0.686022  1.614493
TDL SISO    10db        MLP    0.3406   0.331365  0.3406  0.332320 0.698259  1.598632
TDL SISO    15db        MLP    0.3452   0.333610  0.3452  0.334679 0.698025  1.647198
TDL SISO -3-20db        kNN    0.2796   0.277776  0.2796  0.274924 0.616286  9.644420
TDL SISO     0db        kNN    0.2628   0.270882  0.2628  0.259994 0.581569 10.755055
TDL SISO     5db        kNN    0.2794   0.284158  0.2794  0.277824 0.611216  9.453223
TDL SISO    10db        kNN    0.2992   0.295249  0.2992  0.293934 0.633237  9.085658
TDL SISO    15db        kNN    0.3190   0.310825  0.3190  0.309113 0.644831  9.115723
TDL SISO -3-20db        SVM    0.3210   0.315995  0.32

In [7]:

# ---- TDL CDL SISO ----
print("TDL CDL SISO")
tdl_siso_results = []

df_full = df_tdl_cdl_siso
df_full_test_list = [
    ("-3-20db", df_tdl_cdl_siso_test),
    ("0db", df_tdl_cdl_0db_siso_test),
    ("5db", df_tdl_cdl_5db_siso_test),
    ("10db", df_tdl_cdl_10db_siso_test),
    ("15db", df_tdl_cdl_15db_siso_test),
]

features, labels = preprocess_data_channel(df_full)
features_test, labels_test = preprocess_data_channel(df_full_test_list[0][1])

for clf_name, clf_func in classifiers.items():
    y_train, y_train_pred, y_train_prob, y_test, y_test_pred, y_test_prob, clf = clf_func(features, labels, features_test, labels_test)
    metrics = get_metrics_multiclass(y_test, y_test_pred, y_test_prob)
    metrics['SNR'] = df_full_test_list[0][0]
    metrics['Classifier'] = clf_name
    tdl_siso_results.append(metrics)
    for snr_name, df_snr in df_full_test_list[1:]:
        features_snr, labels_snr = preprocess_data_channel(df_snr)
        # check clf is not train_mlp_channel
        if clf_name != "MLP":
            y_test_pred = clf.predict(features_snr)
            y_test_prob = clf.predict_proba(features_snr) if hasattr(clf, "predict_proba") else None
        else:
            y_test_pred = clf.predict(features_snr).argmax(axis=1)
            y_test_prob = clf.predict(features_snr)
        metrics = get_metrics_multiclass(labels_snr, y_test_pred, y_test_prob)
        metrics['SNR'] = snr_name
        metrics['Classifier'] = clf_name
        tdl_siso_results.append(metrics)

df_tdl_siso_results = pd.DataFrame(tdl_siso_results)
df_tdl_siso_results = df_tdl_siso_results[['Classifier', 'SNR'] + [col for col in df_tdl_siso_results.columns if col not in ['Classifier', 'SNR']]]



# ---- TDL CDL MIMO ----
print("\nTDL CDL MIMO")
tdl_mimo_results = []

df_full = df_tdl_cdl_mimo
df_full_test_list = [
    ("-3-20db", df_tdl_cdl_mimo_test),
    ("0db", df_tdl_cdl_0db_mimo_test),
    ("5db", df_tdl_cdl_5db_mimo_test),
    ("10db", df_tdl_cdl_10db_mimo_test),
    ("15db", df_tdl_cdl_15db_mimo_test),
]

features, labels = preprocess_data_channel(df_full)
features_test, labels_test = preprocess_data_channel(df_full_test_list[0][1])

for clf_name, clf_func in classifiers.items():
    y_train, y_train_pred, y_train_prob, y_test, y_test_pred, y_test_prob, clf = clf_func(features, labels, features_test, labels_test)
    metrics = get_metrics_multiclass(y_test, y_test_pred, y_test_prob)
    metrics['SNR'] = df_full_test_list[0][0]
    metrics['Classifier'] = clf_name
    tdl_mimo_results.append(metrics)
    for snr_name, df_snr in df_full_test_list[1:]:
        features_snr, labels_snr = preprocess_data_channel(df_snr)
        if clf_name != "MLP":
            y_test_pred = clf.predict(features_snr)
            y_test_prob = clf.predict_proba(features_snr) if hasattr(clf, "predict_proba") else None
        else:
            y_test_pred = clf.predict(features_snr).argmax(axis=1)
            y_test_prob = clf.predict(features_snr)
        metrics = get_metrics_multiclass(labels_snr, y_test_pred, y_test_prob)
        metrics['SNR'] = snr_name
        metrics['Classifier'] = clf_name
        tdl_mimo_results.append(metrics)

df_tdl_mimo_results = pd.DataFrame(tdl_mimo_results)
df_tdl_mimo_results = df_tdl_mimo_results[['Classifier', 'SNR'] + [col for col in df_tdl_mimo_results.columns if col not in ['Classifier', 'SNR']]]


/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_abs"] = features["mean"].apply(lambda x: abs(complex(x)))
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_phase"] = features["mean"].apply(lambda x: cmath.phase(complex(x)))    # remove the original mean column


TDL CDL SISO
Epoch 1/1000


/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_abs"] = features["mean"].apply(lambda x: abs(complex(x)))
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_phase"] = features["mean"].apply(lambda x: cmath.phase(complex(x)))    # remove the original mean column
/home/l2nguyen/miniconda3/envs/mlp/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarn

608/625 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1682 - loss: 2.1575
Epoch 1: saving model to mlp_checkpoints/mlp_model_epoch_01.h5


625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.1687 - loss: 2.1557 - val_accuracy: 0.1981 - val_loss: 2.0466 - learning_rate: 0.0010
Epoch 2/1000
606/625 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1988 - loss: 2.0307
Epoch 2: saving model to mlp_checkpoints/mlp_model_epoch_02.h5


...

/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_abs"] = features["mean"].apply(lambda x: abs(complex(x)))
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_phase"] = features["mean"].apply(lambda x: cmath.phase(complex(x)))    # remove the original mean column


313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 958us/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 963us/step


/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_abs"] = features["mean"].apply(lambda x: abs(complex(x)))
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_phase"] = features["mean"].apply(lambda x: cmath.phase(complex(x)))    # remove the original mean column
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to b

Performing cross-validation...
Cross-Validation Accuracy: 0.2618 ± 0.0033


/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_abs"] = features["mean"].apply(lambda x: abs(complex(x)))
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_phase"] = features["mean"].apply(lambda x: cmath.phase(complex(x)))    # remove the original mean column
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to b

In [8]:
print(df_tdl_siso_results.to_string(index=False))
print(df_tdl_mimo_results.to_string(index=False))
# save the results to a csv and xlsx file
df_tdl_siso_results.to_csv("results_tdl_siso.csv", index=False)
# df_tdl_siso_results.to_excel("results_tdl_siso.xlsx", index=False)
df_tdl_mimo_results.to_csv("results_tdl_mimo.csv", index=False)
# df_tdl_mimo_results.to_excel("results_tdl_mimo.xlsx", index=False)

Classifier     SNR  Accuracy  Precision  Recall  F1 Score  ROC AUC  Log Loss
       MLP -3-20db    0.2283   0.225405  0.2283  0.223736 0.744047  2.025938
       MLP     0db    0.2139   0.209841  0.2139  0.203443 0.716551  2.069370
       MLP     5db    0.2135   0.212134  0.2135  0.207422 0.730830  2.023964
       MLP    10db    0.2469   0.249989  0.2469  0.238787 0.756231  1.984040
       MLP    15db    0.2656   0.261294  0.2656  0.259708 0.770598  1.958938
       kNN -3-20db    0.1724   0.169430  0.1724  0.166688 0.609011 16.878392
       kNN     0db    0.1492   0.150158  0.1492  0.141758 0.571266 18.690877
       kNN     5db    0.1582   0.153807  0.1582  0.149687 0.593645 17.463623
       kNN    10db    0.1979   0.198506  0.1979  0.193191 0.629339 15.864971
       kNN    15db    0.2076   0.202911  0.2076  0.201976 0.646703 15.185795
       SVM -3-20db    0.1973   0.215510  0.1973  0.169220 0.715560  2.024393
       SVM     0db    0.1660   0.187120  0.1660  0.124030 0.671958  2.135846

SIONNA MIMO Dataset NLOS/LOS Classification
=======================================

In [ ]:
# # Train the model and get predictions
# features_sionna_mimo, labels_sionna_mimo = preprocess_data_channel(df_sionna_mimo)
# features_sionna_mimo_test, labels_sionna_mimo_test = preprocess_data_channel(df_sionna_mimo_test)

# y_sionna_mimo_train, y_sionna_mimo_train_pred, y_sionna_mimo_train_prob, y_sionna_mimo_test, y_sionna_mimo_test_pred, y_sionna_mimo_test_prob = train_mlp_channel(features_sionna_mimo, labels_sionna_mimo, features_sionna_mimo_test, labels_sionna_mimo_test, model_name='mlp_model_sionna_mimo.keras')
# # Plot confusion matrix and ROC curve for sionna mimo training set
# class_labels = df_sionna_mimo["scenario"].astype(str)
# class_labels = class_labels.unique()
# plot_confusion_matrix_multiclass(y_sionna_mimo_train, y_sionna_mimo_train_pred, class_labels, title='Confusion Matrix - Sionna MIMO Training Set')
# plot_roc_curve_multiclass(y_sionna_mimo_train, y_sionna_mimo_train_prob, class_labels, title='ROC Curve - Sionna MIMO Training Set')

# # Print metrics for sionna mimo training set
# print("Metrics for Sionna MIMO Training Set:")
# plot_confusion_matrix_multiclass(y_sionna_mimo_train, y_sionna_mimo_train_pred, class_labels, title='Confusion Matrix - Sionna MIMO Training Set')
# plot_roc_curve_multiclass(y_sionna_mimo_train, y_sionna_mimo_train_prob, class_labels, title='ROC Curve - Sionna MIMO Training Set')

# # Plot confusion matrix and ROC curve for sionna mimo test set
# plot_confusion_matrix_multiclass(y_sionna_mimo_test, y_sionna_mimo_test_pred, class_labels, title='Confusion Matrix - Sionna MIMO Test Set')
# plot_roc_curve_multiclass(y_sionna_mimo_test, y_sionna_mimo_test_prob, class_labels, title='ROC Curve - Sionna MIMO Test Set')

# # Print metrics for sionna mimo test set
# print("Metrics for Sionna MIMO Test Set:")
# print_metrics_multiclass(y_sionna_mimo_test, y_sionna_mimo_test_pred, y_sionna_mimo_test_prob)

SIONNA SISO Dataset NLOS/LOS Classification
=======================================

In [ ]:
# # Load the datasets
# df_siso = pd.read_csv("dataset_sionna_siso.csv")

# # Combine the datasets
# df = df_siso
# # take 80 percent of the data for training, 20 percent for testing
# features, labels = preprocess_data_channel(df.sample(frac=0.8))
# features_test, labels_test = preprocess_data_channel(df.sample(frac=0.2))
# # Train the model and get predictions
# y_train, y_train_pred, y_train_prob, y_test, y_test_pred, y_test_prob = train_mlp_channel(features, labels, features_test, labels_test, model_name='mlp_model_siso.keras')

# # Plot confusion matrix and ROC curve for training set
# class_labels = df["scenario"].astype(str)
# class_labels = class_labels.unique()
# plot_confusion_matrix_multiclass(y_train, y_train_pred, class_labels, title='Confusion Matrix - Training Set')
# plot_roc_curve_multiclass(y_train, y_train_prob, class_labels, title='ROC Curve - Training Set')


# # Print metrics for training set
# print("Metrics for Training Set:")
# print_metrics_multiclass(y_train, y_train_pred, y_train_prob)

# # Plot confusion matrix and ROC curve for test set
# class_labels = df["scenario"].astype(str)
# class_labels = class_labels.unique()
# plot_confusion_matrix_multiclass(y_test, y_test_pred, class_labels, title='Confusion Matrix - Test Set')
# plot_roc_curve_multiclass(y_test, y_test_prob, class_labels, title='ROC Curve - Test Set')

# # Print metrics for test set
# print("Metrics for Test Set:")
# print_metrics_multiclass(y_test, y_test_pred, y_test_prob)